
# 기상 데이터 기반 내일 비 예측 머신러닝 실습

##  목표
오늘의 기상 데이터를 이용하여 **내일 비가 오는지(0/1)** 예측하는 머신러닝 수행


## 사용 데이터 :
- `weather_2020_2025.csv` : 서울(108) 일별 기상 관측 2020~2025 

### 입력 변수(X)
- avgTa : 평균기온
- minTa : 최저기온
- maxTa : 최고기온
- avgRhm : 평균습도
- avgWs : 평균풍속
- sumRn : 강수량
- avgPa : 평균기압
- sumSsHr : 일조시간
- avgTca : 전운량
- month : 월 정보

### 타겟 변수(y)
- tomorrow_rain
    - 1 : 내일 비가 옴
    - 0 : 내일 비가 오지 않음


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 분류 모델
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 회귀 모델
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# 평가 지표
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)


## 1. 필요한 컬럼만 선택

실무에서는 원본 데이터 전체를 사용하는 것보다,
필요한 컬럼만 선택하여 사용하는 것이 좋다.

장점:
- 메모리 절약
- 코드 가독성 향상
- 불필요 변수 제거
- 모델 안정성 향상


In [2]:
df = pd.read_csv('weather_2020_2025.csv')

In [3]:
df = df[['tm', 'avgTa', 'minTa', 'maxTa', 'avgRhm','avgWs', 'sumRn', 'avgPa', 
      'sumSsHr', 'avgTca']].copy()

## 2. 날짜형 변환

시계열 데이터 분석에서는 날짜형 변환이 중요하다.


In [4]:
df['tm']

0       2020-01-01
1       2020-01-02
2       2020-01-03
3       2020-01-04
4       2020-01-05
           ...    
2187    2025-12-27
2188    2025-12-28
2189    2025-12-29
2190    2025-12-30
2191    2025-12-31
Name: tm, Length: 2192, dtype: object

In [5]:
df['tm'] = pd.to_datetime(df['tm'])

In [6]:
df['tm']

0      2020-01-01
1      2020-01-02
2      2020-01-03
3      2020-01-04
4      2020-01-05
          ...    
2187   2025-12-27
2188   2025-12-28
2189   2025-12-29
2190   2025-12-30
2191   2025-12-31
Name: tm, Length: 2192, dtype: datetime64[ns]

## 3. month feature 생성

계절성과 월별 패턴을 학습하기 위해 month 변수를 추가한다.

In [7]:
df['month'] = df['tm'].dt.month

## 4. rain 컬럼 생성

강수량(sumRn)이 0보다 크면 비가 온 것으로 판단한다.

- 비가 오면 1
- 비가 안 오면 0

In [8]:
df.isnull().sum()

tm            0
avgTa         0
minTa         1
maxTa         0
avgRhm        0
avgWs         5
sumRn      1266
avgPa         1
sumSsHr       7
avgTca        0
month         0
dtype: int64

In [9]:
df = df.fillna(0)

In [10]:
def make_rain(x):
    if x > 0:
        return 1
    else:
        return 0

In [11]:
df['rain']=df['sumRn'].apply(make_rain)

In [12]:
df['rain'].value_counts()

rain
0    1529
1     663
Name: count, dtype: int64

In [13]:
df.columns

Index(['tm', 'avgTa', 'minTa', 'maxTa', 'avgRhm', 'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca', 'month', 'rain'],
      dtype='object')


## 5. shift(-1)을 이용한 내일 비 생성

머신러닝에서 중요한 점은:

> 오늘 데이터로 내일을 예측해야 한다.

shift(-1)은 데이터를 위로 한 칸 이동한다.

예:
- 오늘 행에 내일 비 정보가 들어감


In [14]:
df['tomorrow_rain'] = df['rain'].shift(-1)

In [15]:
df.tail()

,tm,avgTa,minTa,maxTa,avgRhm,avgWs,sumRn,avgPa,sumSsHr,avgTca,month,rain,tomorrow_rain
2187,2025-12-27,-2.6,-8.3,0.5,52.1,2.1,0.0,1015.3,2.0,7.8,12,0,1.0
2188,2025-12-28,3.2,0.0,6.1,76.4,2.0,0.6,1011.1,0.7,8.3,12,1,1.0
2189,2025-12-29,4.9,-0.1,9.1,74.1,2.1,0.3,1008.0,4.9,3.1,12,1,0.0
2190,2025-12-30,-0.5,-3.7,3.8,45.1,2.4,0.0,1015.1,9.0,1.9,12,0,0.0
2191,2025-12-31,-5.6,-8.9,-1.2,40.9,3.1,0.0,1018.0,9.0,0.3,12,0,NaN


## 6. NaN 제거

마지막 행은 다음날 데이터가 없기 때문에 NaN이 발생한다.


In [16]:
df = df.dropna(subset=['tomorrow_rain'])


## 7. Feature / Target 분리


In [17]:
df.columns

Index(['tm', 'avgTa', 'minTa', 'maxTa', 'avgRhm', 'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca', 'month', 'rain', 'tomorrow_rain'],
      dtype='object')

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2191 entries, 0 to 2190
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   tm             2191 non-null   datetime64[ns]
 1   avgTa          2191 non-null   float64       
 2   minTa          2191 non-null   float64       
 3   maxTa          2191 non-null   float64       
 4   avgRhm         2191 non-null   float64       
 5   avgWs          2191 non-null   float64       
 6   sumRn          2191 non-null   float64       
 7   avgPa          2191 non-null   float64       
 8   sumSsHr        2191 non-null   float64       
 9   avgTca         2191 non-null   float64       
 10  month          2191 non-null   int32         
 11  rain           2191 non-null   int64         
 12  tomorrow_rain  2191 non-null   float64       
dtypes: datetime64[ns](1), float64(10), int32(1), int64(1)
memory usage: 231.1 KB


In [19]:
df['tomorrow_rain'] = df['tomorrow_rain'].astype(int)

In [20]:
X = df[['avgTa', 'minTa', 'maxTa', 'avgRhm', 'avgWs', 'sumRn', 'avgPa',
       'sumSsHr', 'avgTca', 'month' ]]
y = df['tomorrow_rain']

## 8. train / test split

### stratify=y 의미

클래스 비율을 유지하면서 train/test 데이터를 분리한다. (분류에서만 사용)

예:
- 비 오는 날 비율
- 비 안 오는 날 비율 을 train/test 모두 비슷하게 유지한다.


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)


## 9. 모델 학습
### class_weight='balanced'

비 오는 날과 안 오는 날의 데이터 개수가 다를 수 있기 때문에
클래스 가중치를 자동 조정한다


In [22]:
rf = RandomForestClassifier(random_state=0, class_weight='balanced')
rf

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
y_pred = rf.predict(X_test)

## 10. 모델 평가

- Accuracy : 정확도
- Precision : 비 온다고 예측한 것 중 실제 비인 비율
- Recall : 실제 비 온 날 중 맞춘 비율
- F1-score : Precision과 Recall의 조화 평균

In [25]:
confusion_matrix(y_test,y_pred)

array([[277,  29],
       [ 85,  48]])

In [26]:
acc_rf = accuracy_score(y_test, y_pred)
precision_rf  = precision_score(y_test, y_pred)
recall_rf  = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
print(f'정확도 (Accuracy): {acc_rf :.2f}')      
print(f'정밀도 (Precision): {precision_rf :.2f}') 
print(f'재현율 (Recall): {recall_rf :.2f}')       
print(f'f1-score: {f1_rf:.2f}')                  

정확도 (Accuracy): 0.74
정밀도 (Precision): 0.62
재현율 (Recall): 0.36
f1-score: 0.46


In [27]:
from sklearn.metrics import classification_report

In [28]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.91      0.83       306
           1       0.62      0.36      0.46       133

    accuracy                           0.74       439
   macro avg       0.69      0.63      0.64       439
weighted avg       0.72      0.74      0.72       439



- 시계열 분석은 과거 데이터를 이용하여 미래를 예측하는 머신러닝 방법     
> 주가, 사용량, 수요 등의 예측에 활용